# Schema evolution — a rule that fails open

Replaces `scripts/demo_fail_open.py`.

The reviewer's third objection was *"no schema evolution"*. The point is not that identifiers change — that is obvious — but that a rule depending on one **stops protecting anything, silently**.

## Setup

Redacted readiness report — `config.py` never prints a secret.

In [1]:
import sys, warnings
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")          # Neo4j emits schema hints we handle
import config
print(config.report())

readiness
  neo4j      uri=neo4j+s://<redacted>.databases.neo4j.io
             user fp=6b611cde  db fp=6b611cde
             password fp=58ebef03
             configured=True
  qdrant     url=https://<redacted>.sa-east-1-0.aws.cloud.qdrant.io
             api_key fp=f3da5a0f
             configured=True
  embedding  provider=bedrock  model=amazon.titan-embed-text-v2:0  dim=1024
             configured=True
  trust      policy=v1.0  floor=0.75  min_tier=ingested


## Real HGNC renames, read back from the graph

In [2]:
from biomedical.graph.release_diff import diff_from_graph
from biomedical.graph.store import GraphClient
cfg = config.load()

with GraphClient(uri=cfg.neo4j.uri, username=cfg.neo4j.username,
                 password=cfg.neo4j.password.reveal(), database=cfg.neo4j.database) as c:
    diff = diff_from_graph(c)
    current = {r["s"] for r in c.run("MATCH (g:Gene) RETURN g.symbol AS s")}
    trials = {r["id"]: {"phase": r["phase"]} for r in
              c.run("MATCH (t:ClinicalTrial) RETURN t.nct_id AS id, t.phase AS phase LIMIT 50")}

print(diff.describe())
print("\norphan obsoletions (no automatic path forward):", len(diff.orphans()))

hgnc-symbol: historical -> current
  7 changed identifier(s)  {'renamed': 7}
  hgnc-symbol:CDKN2 -> CDKN2A [renamed] HGNC:1787
  hgnc-symbol:MLM -> CDKN2A [renamed] HGNC:1787
  hgnc-symbol:PBT -> KIT [renamed] HGNC:6342
  hgnc-symbol:WS2 -> MITF [renamed] HGNC:7105
  hgnc-symbol:WS2A -> MITF [renamed] HGNC:7105
  hgnc-symbol:BZS -> PTEN [renamed] HGNC:9588
  hgnc-symbol:MHAM -> PTEN [renamed] HGNC:9588

orphan obsoletions (no automatic path forward): 0


## The rule under test

Authored **2016-04-02** against `CDKN2`, which HGNC has since renamed to `CDKN2A`. The rule has not been touched.

In [3]:
from biomedical.graph.rule_integrity import (EvaluationContext, check_integrity,
                                             evaluate_rule, load_rules, propose_remediations)

rules = load_rules()
for r in rules:
    deps = ", ".join(f"{a}:{v}" for a, v in r.dependencies()) or "-"
    print(f"  {r.rule_id:<48} authored {r.authored}  depends on {deps}")

rule = next(r for r in rules if r.rule_id == "safety.cdkn2_loss_requires_genotyping")
findings = check_integrity(rules, diff)

  safety.cdkn2_loss_requires_genotyping            authored 2016-04-02  depends on hgnc-symbol:CDKN2
  safety.mitf_amplification_disclosure             authored 2015-11-18  depends on hgnc-symbol:WS2A
  safety.braf_mutation_requires_genotyping         authored 2024-01-10  depends on hgnc-symbol:BRAF
  governance.trial_still_recruiting                authored 2023-06-01  depends on -
  integrity.rule_cites_obsolete_identifier         authored None  depends on -


## Three runs. Same rule, same code, same facts.

Only the vocabulary release differs.

In [4]:
historical = set(current) | set(diff.moved_identifiers())
before = EvaluationContext(gene_symbols=historical, trial_properties=trials)
after = EvaluationContext(gene_symbols=current, trial_properties=trials)

r1 = evaluate_rule(rule, before)
r2 = evaluate_rule(rule, after)                       # v1 behaviour
r3 = evaluate_rule(rule, after, integrity=findings)   # the fix

for label, res, note in [
    ("release 'historical'", r1, "correct"),
    ("release 'current', unchecked", r2, "*** FAIL-OPEN ***"),
    ("release 'current', checked", r3, "correct"),
]:
    print(f"{label:<32} {res.outcome.value.upper():<16} {note}")
    print(f"    {res.message}")
    if res.missing_dependencies:
        print(f"    withheld dependency: {', '.join(res.missing_dependencies)}")

print("\nHGNC renamed CDKN2 -> CDKN2A. Unchecked, the rule matched nothing and")
print("reported NO_VIOLATION — indistinguishable from 'checked and clean'.")
print("No error. No alert. The control simply stopped.")

release 'historical'             VIOLATION        correct
    gene_symbol=CDKN2 present and 'genotyping_plan' absent
release 'current', unchecked     NO_VIOLATION     *** FAIL-OPEN ***
    no entity matched gene_symbol=CDKN2
release 'current', checked       CANNOT_EVALUATE  correct
    dependency hgnc-symbol:CDKN2 was renamed to CDKN2A
    withheld dependency: hgnc-symbol:CDKN2

HGNC renamed CDKN2 -> CDKN2A. Unchecked, the rule matched nothing and
reported NO_VIOLATION — indistinguishable from 'checked and clean'.
No error. No alert. The control simply stopped.


## Every rule the current release breaks

Remediation is **proposed, never applied**. Following a rename automatically would be the unaudited inference this layer exists to prevent.

In [5]:
for f in findings:
    print(" ", f.describe())
    print(f"     rule authored {f.authored}; recoverable: {f.recoverable}")

print("\nproposals (pending, not applied):")
for p in propose_remediations(findings):
    print("  ", p.describe())

  safety.cdkn2_loss_requires_genotyping depends on hgnc-symbol:CDKN2, which was renamed -> CDKN2A
     rule authored 2016-04-02; recoverable: True
  safety.mitf_amplification_disclosure depends on hgnc-symbol:WS2A, which was renamed -> MITF
     rule authored 2015-11-18; recoverable: True

proposals (pending, not applied):
   safety.cdkn2_loss_requires_genotyping: propose CDKN2 -> CDKN2A [pending]
   safety.mitf_amplification_disclosure: propose WS2A -> MITF [pending]


## The CI gate

Runs offline against a committed fixture, so it gates a build with no credentials. A notebook that raises fails execution, which preserves the gate semantics the script had via its exit code.

In [6]:
import json, pathlib
from biomedical.graph.release_diff import ChangeKind, ReleaseDiff, SymbolChange

d = json.loads(pathlib.Path("../tests/fixtures/hgnc_release_diff.json").read_text())
fixture = ReleaseDiff(
    authority=d["authority"], release_from=d["release_from"], release_to=d["release_to"],
    changes=tuple(SymbolChange(d["authority"], c["old_value"], c["new_value"],
                               ChangeKind(c["change_kind"]), c.get("stable_id"),
                               c.get("entity_name")) for c in d["changes"]))

print("fixture note:", d["_note"][:96], "...")
offline = check_integrity(load_rules(), fixture)
print(f"\nrules checked: {len(load_rules())}   identifiers moved: {len(fixture.moved_identifiers())}")
print(f"CI verdict: {'FAIL' if offline else 'PASS'} — {len(offline)} rule(s) depend on a moved identifier")
for f in offline:
    print("  ", f.rule_id)

fixture note: Captured from the live HGNC-derived graph diff. The final entry is a synthetic orphan so the unr ...

rules checked: 5   identifiers moved: 8
CI verdict: FAIL — 2 rule(s) depend on a moved identifier
   safety.cdkn2_loss_requires_genotyping
   safety.mitf_amplification_disclosure
